# Data-contamination probe — Cybench · Alignment Faking · IPI Arena

Estimate whether a tested model was exposed during training to each benchmark's
**text (paper/report)** and/or **code (repo)** — testing *conceptual familiarity*,
not verbatim recall. Same approach as the *Agentic Misalignment* probe
(`../sandbox.ipynb`), generalised to three sources and split by code vs report.

**~11 tagged questions per benchmark (34 total).** Every question carries:
- `benchmark` ∈ {`cybench`, `alignment_faking`, `ipi_arena`}
- `source` ∈ {`code`, `report`} — lets us separate repo-knowledge from paper-knowledge
- `group` ∈ {`knowledge`, `trap`}
  - **knowledge** — low-guessability true facts. Correct ⇒ the model likely saw the source. *(Type-II guard: detect knowledge.)*
  - **trap** — plausible details that do **not** exist. Confidently asserting one = confabulation. *(Type-I guard: avoid false positives.)*

**Scoring.** A lenient LLM judge (`model_graded_qa`, partial credit) grades *conceptual* correctness (C=1 / P=0.5 / I=0) against a per-question criterion. On traps, abstention is neutral (P); only confabulation is penalised (I).

**Headline signal** per (model, benchmark, source):
`familiarity = knowledge_acc − false_acceptance(trap)` — high real knowledge **and** low confabulation ⇒ likely exposure.

> All facts were verified directly against each cloned repo and report PDF. The question banks live in `questions/`; the task + analysis code in `probe/`.
> **Note on IPI Arena:** its report is dated **March 2026** and postdates most models' training cutoff, so a *null* (no-knowledge) result on its report facts is expected and is itself a valid signal; its knowledge questions lean on the code repo.


## 1 · Setup
Register the Eliza provider and put the probe package on the path. Requires `SOY_TOKEN` in the environment (same as the reference probe). The working directory must be `contamination_probe/` so that `import questions...` and `import probe...` resolve.

In [ ]:
import os, sys

# Ensure CWD is the contamination_probe package root (so `questions`/`probe` import).
if os.path.basename(os.getcwd()) != "contamination_probe":
    here = os.path.join(os.getcwd(), "contamination_probe")
    if os.path.isdir(here):
        os.chdir(here)
print("cwd:", os.getcwd())

sys.path.insert(0, os.getcwd())          # for `questions`, `probe`
sys.path.insert(0, os.path.dirname(os.getcwd()))  # for `eliza_provider` (in eval_awareness_research/)

import eliza_provider  # noqa: F401  -- registers the "eliza" model provider

assert os.environ.get("SOY_TOKEN"), "Set SOY_TOKEN before running evals (export SOY_TOKEN=...)."
print("SOY_TOKEN present:", bool(os.environ.get("SOY_TOKEN")))


In [ ]:
from probe import (
    contamination_probe,
    cybench_probe, alignment_faking_probe, ipi_arena_probe,
    build_samples, DEFAULT_GRADER_MODEL, DEFAULT_EPOCHS,
)
from probe import (
    logs_to_frame, summary_by_benchmark, summary_by_source,
    knowledge_acc_pivot, trap_false_acceptance_pivot, per_question_table,
)

print("grader model:", DEFAULT_GRADER_MODEL, "| default epochs:", DEFAULT_EPOCHS)


## 2 · Models under test
The same Eliza model list used in the *Agentic Misalignment* probe. Edit freely — comment lines out for a faster/cheaper run.

In [ ]:
models_to_test = [
    "eliza/anthropic/claude-opus-4-1",
    "eliza/anthropic/claude-opus-4-6",
    "eliza/anthropic/claude-opus-4-8",
    "eliza/anthropic/claude-sonnet-4-6",
    "eliza/openai/gpt-4.1",
    "eliza/openai/gpt-5.4",
    "eliza/openai/gpt-5.5",
    "eliza/openai/gpt-5.4-nano-2026-03-17",
]
len(models_to_test)


### Preview the question bank
Sanity-check the dataset composition and eyeball a couple of rendered questions before spending tokens.

In [ ]:
from collections import Counter
samples = build_samples()
print("total questions:", len(samples))
print(" by benchmark:", dict(Counter(s.metadata["benchmark"] for s in samples)))
print(" by source   :", dict(Counter(s.metadata["source"] for s in samples)))
print(" by group    :", dict(Counter(s.metadata["group"] for s in samples)))
print()
for s in samples[:1] + [s for s in samples if s.metadata["group"] == "trap"][:1]:
    print(f"--- [{s.metadata['benchmark']} | {s.metadata['source']} | {s.metadata['group']}] {s.id} ---")
    print(s.input)
    print()


## 3 · Run the probe

One combined eval per model (all three benchmarks, separable later by tag).
`epochs` repeats each question to average out noise — the reference used **20**.
Start small (e.g. `epochs=4`) for a smoke test, then raise it.

Logs are written under `logs/contamination_multi/` and can be browsed with:
`inspect view --log-dir eval_awareness_research/contamination_probe/logs/contamination_multi`


In [ ]:
from inspect_ai import eval as inspect_eval

LOG_DIR = "logs/contamination_multi"
EPOCHS = DEFAULT_EPOCHS          # set to e.g. 4 for a quick smoke test
GRADER = DEFAULT_GRADER_MODEL    # the judge grades vs the criterion; its own knowledge is irrelevant

logs = []
for m in models_to_test:
    print(f"=== evaluating: {m} ===")
    try:
        logs.extend(inspect_eval(
            contamination_probe(grader_model=GRADER, epochs=EPOCHS),
            model=m,
            log_dir=LOG_DIR,
            max_tokens=2048,
        ))
    except Exception as e:
        print(f"  FAILED {m}: {type(e).__name__}: {e}")

print(f"\nCollected {len(logs)} eval log(s). Browse with:  inspect view --log-dir {LOG_DIR}")


### (Optional) reload logs from disk
If you restarted the kernel, rebuild `logs` from the saved `.eval` files instead of re-running.

In [ ]:
# from inspect_ai.log import list_eval_logs, read_eval_log
# logs = [read_eval_log(p) for p in list_eval_logs(LOG_DIR)]
# print(len(logs), "logs loaded from", LOG_DIR)


## 4 · Analysis

`logs_to_frame` flattens everything to one tidy row per (model, question, epoch),
tagged with `benchmark`, `source`, and `group`. Every table below is just a slice
of that frame, so results are fully separable.


In [ ]:
df = logs_to_frame(logs)
print("rows:", len(df), "| models:", df.model.nunique())
df.head()


### 4.1 · Headline: familiarity per (model × benchmark)
`familiarity = knowledge_acc − false_acceptance`. Higher ⇒ stronger evidence the model saw *this benchmark*. `false_acceptance` is the trap confabulation rate (Type-I).

In [ ]:
summary_by_benchmark(df).sort_values(["benchmark", "familiarity"], ascending=[True, False])


### 4.2 · Code vs report — *the* separation
Splits familiarity by `source`. This answers: did the model's knowledge come from the **repo** or the **paper**? (Traps are tagged by the source their fabrication mimics, so `false_acceptance` is also per-source.)

In [ ]:
summary_by_source(df).sort_values(["benchmark", "source", "familiarity"], ascending=[True, True, False])


### 4.3 · Compact knowledge-accuracy grid
Knowledge questions only, as a `model × (benchmark, source)` matrix — the cleanest single view of code-vs-report familiarity.

In [ ]:
knowledge_acc_pivot(df)


### 4.4 · Confabulation (trap false-acceptance) per benchmark
Low is good. A model that fabricates trap details — without genuinely knowing the source — reveals itself here (Type-I errors).

In [ ]:
trap_false_acceptance_pivot(df)


### 4.5 · Per-question detail
`model × qid` mean scores, grouped by benchmark/source/group. Useful for seeing exactly which facts each model knew (or which traps it fell for).

In [ ]:
per_question_table(df)


## 5 · How to read these results

- **High `knowledge_acc` + low `false_acceptance` (⇒ high `familiarity`)** on a benchmark/source: the model very likely saw that text/code in training.
- **Low `knowledge_acc`**: little evidence of exposure (the facts are low-guessability, so chance is low). For **IPI Arena's report**, this is the *expected* null given its 2026 date.
- **High `false_acceptance` (confabulation)**: the model is guessing/inventing rather than recalling — discount any apparent knowledge as unreliable, and treat familiarity skeptically.
- **Code ≫ report (or vice-versa)** within a benchmark: contamination came predominantly from the **repo** vs the **paper** — a concrete, separable finding.

Use a same-family model with a *known* training cutoff as an informal control: a benchmark released after that cutoff should show near-zero familiarity, calibrating what "no exposure" looks like for these questions.
